In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests
import pandas as pd
import numpy as np
import os

In [ ]:
# Input and output folders for Excel files
data_folder = "Data/FilteredData"
output_folder = "Data/PreparedMatrixAdjacency"

# Granger causality test configuration
test = 'ssr_chi2test'
maxlag = 1

In [ ]:
# Create output folder if it does not exist
os.makedirs(output_folder, exist_ok=True)

In [ ]:
def grangers_causation_matrix(data, variables, test='ssr_chi2test', maxlag=1, verbose=False):
    """
    Compute Granger causality matrix for all variable pairs.
    
    Rows represent response variables, columns represent predictors.
    Values are p-values. A p-value < 0.05 indicates rejection of the null hypothesis
    (i.e., the predictor Granger-causes the response).

    data      : pandas DataFrame with time series data
    variables : list of variable names
    test      : test statistic to use (default: 'ssr_chi2test')
    maxlag    : maximum lag to consider (default: 1)
    verbose   : print detailed results if True
    """
    
    df = pd.DataFrame(
        np.zeros((len(variables), len(variables))),
        columns=variables,
        index=variables
    )
    
    for c in df.columns:
        for r in df.index:
            try:
                test_result = grangercausalitytests(
                    data[[r, c]],
                    maxlag=maxlag,
                    verbose=False
                )
                
                p_values = [
                    round(test_result[i+1][0][test][1], 4)
                    for i in range(maxlag)
                ]
                
                min_p_value = np.min(p_values)
                df.loc[r, c] = min_p_value
            
            except Exception:
                # If an error occurs, assign 0
                df.loc[r, c] = 0
    
    df.columns = variables
    df.index = variables
    
    return df

In [ ]:
# Iterate over Excel files in the folder
for filename in os.listdir(data_folder):
    if filename.endswith(".xlsx"):
        file_path = os.path.join(data_folder, filename)
        
        # Extract date from filename
        date_str = os.path.splitext(filename)[0].split('_')[-1]
        
        # Load Excel file
        df = pd.read_excel(file_path)
        
        # Drop columns with all missing values
        df = df.dropna(axis=1, how='all')
        
        # Remove date column
        columns_to_exclude = ['Date']
        df = df.drop(columns_to_exclude, axis=1)
        
        # Remove constant columns
        constant_columns = [col for col in df.columns if df[col].nunique() == 1]
        df = df.drop(constant_columns, axis=1)
        
        # Compute Granger causality matrix
        gc_matrix = grangers_causation_matrix(df, variables=df.columns, verbose=True)
        
        # Output file path
        output_file = os.path.join(output_folder, f"matrix_{date_str}.xlsx")
        
        # Save matrix to Excel
        gc_matrix.to_excel(output_file, index=True)
        
        # Print status message
        print(f"Granger causality matrix for {filename} created successfully.")